# 01 – Setup and model check

Run this notebook **once** on Data Lab JupyterHub (H200). It:

1. checks the machine,
2. installs/upgrades the libraries Qwen3-VL needs,
3. downloads the vision-language model, by default **Qwen/Qwen3-VL-30B-A3B-Instruct** (~62 GB, one time),
4. loads it on the GPU and tests a text question and an image question.

The RAG part (index + search + answers with sources) comes in `02_rag_chatbot.ipynb`.

## 1. Check the machine

In [ ]:
import sys, subprocess, importlib

def sh(cmd):
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

print("Python:", sys.version.split()[0])
sh("nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv")
sh("free -h; df -h ~")

for pkg in ["torch", "torchvision", "transformers", "accelerate", "sentence_transformers",
            "faiss", "PIL", "gradio"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:22s} {getattr(m, '__version__', 'installed')}")
    except Exception:
        print(f"{pkg:22s} NOT INSTALLED")

## 2. Install / upgrade libraries

- `transformers` 4.46 is too old for Qwen3-VL, so it is upgraded.
- `torch` is pinned to the installed **2.9.1** so pip doesn't replace the CUDA build.
- Qwen3-VL's processor needs `torchvision`; **0.24.1** is the version that matches torch 2.9.1 (CUDA 12.8).

If you get a *permission denied* error, add `--user` to both lines.

**⚠️ When this cell finishes: menu Kernel → Restart Kernel, then continue from step 3 (don't run step 2 again).**

In [ ]:
%pip install -q -U transformers accelerate sentence-transformers gradio "torch==2.9.1"
%pip install -q "torchvision==0.24.1" --index-url https://download.pytorch.org/whl/cu128

## 3. Confirm versions + choose the model (after the kernel restart)

To use a different vision-language model, change `MODEL_ID` here (and in notebook 02's settings).
Supported choices and their GPU memory are listed in the project README.

In [ ]:
import os, torch, torchvision, transformers, sentence_transformers
from transformers import AutoConfig, AutoModelForImageTextToText

MODEL_ID  = "Qwen/Qwen3-VL-30B-A3B-Instruct"                        # <- change the model here
MODEL_DIR = os.path.expanduser(f"~/hf_models/{MODEL_ID.split('/')[-1]}")

print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("transformers", transformers.__version__, "| sentence-transformers", sentence_transformers.__version__)
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

# fails here if this transformers version doesn't know the model
cls = AutoModelForImageTextToText._model_mapping[type(AutoConfig.from_pretrained(MODEL_ID))]
print(f"{MODEL_ID} -> {cls.__name__}: supported")

## 4. Download the model (one time; ~62 GB for the default model)

Safe to re-run: files already downloaded are skipped. If this fails with a connection error,
the hub has no internet access to Hugging Face.

In [ ]:
import time
from huggingface_hub import snapshot_download

t = time.time()
snapshot_download(MODEL_ID, local_dir=MODEL_DIR)
print(f"Downloaded to {MODEL_DIR} in {(time.time() - t) / 60:.1f} min")

## 5. Load the model onto the GPU

The default model in bf16 (full quality) uses ~62 GB of the H200's 143 GB, leaving room for the embedding
model and long contexts. Takes ~1 min. `dtype="auto"` keeps each model's own precision (bf16, or FP8 for `-FP8` models).

In [ ]:
import time
from transformers import AutoProcessor, AutoModelForImageTextToText

t = time.time()
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_DIR,
    dtype="auto",
    device_map="cuda:0",
    attn_implementation="sdpa",
).eval()
processor = AutoProcessor.from_pretrained(MODEL_DIR)

print(f"Loaded in {time.time() - t:.0f}s | GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 6. Helper functions

- `load_image()` shrinks big phone photos (a 12 MP photo would otherwise cost ~12k tokens and be slow).
  1600 px on the long side keeps notice/timetable text readable (~1,900 tokens).
- `ask_model()` sends a chat to Qwen3-VL and returns the reply, printing speed.

In [ ]:
import time, torch
from PIL import Image

def load_image(path_or_img, max_side=1600):
    img = path_or_img if isinstance(path_or_img, Image.Image) else Image.open(path_or_img)
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img

def ask_model(messages, max_new_tokens=512, verbose=True):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    t = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    if verbose:
        n, dt = new_tokens.shape[1], time.time() - t
        print(f"[{inputs['input_ids'].shape[1]} prompt tokens -> {n} new tokens in {dt:.1f}s = {n / dt:.1f} tok/s]")
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

## 7. Test: text question

In [ ]:
messages = [
    {"role": "user", "content": [
        {"type": "text", "text": "In 3 short bullet points, what kinds of questions can a university help-desk chatbot answer?"},
    ]},
]
print(ask_model(messages, max_new_tokens=200))

## 8. Test: image question

Upload any image with the ⬆ button in the left file panel — a notice, a timetable screenshot, or one of your
campus photos (e.g. `admissionoffice.jpg`) — and put its file name below.

In [ ]:
import os
IMAGE_PATH = "admissionoffice.jpg"   # <- change to your uploaded file

if os.path.exists(IMAGE_PATH):
    img = load_image(IMAGE_PATH)
    display(img.resize((img.width // 3, img.height // 3)))   # small preview
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "Read all visible text in this image, then explain in 2-3 sentences what it shows."},
        ]},
    ]
    print(ask_model(messages, max_new_tokens=300))
else:
    print(f"Upload an image first: {IMAGE_PATH} not found in {os.getcwd()}")

## 9. GPU memory after the tests

In [ ]:
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB | Peak: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB "
      f"| Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")